# FloodLens Group A — Colab data pipeline
Upload the downloadable project ZIP, update the actual API history, and export a single CSV. Uses the same 25 district reference points as the website. No prediction model is fitted. Large downloads are cached; API limits can pause collection. Forecast CSV remains separate from historical values.

In [ ]:
from google.colab import files
import zipfile, pathlib, os
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith(".zip"))
destination = pathlib.Path("/content/floodlens-project")
destination.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        if not (destination / name).resolve().is_relative_to(destination.resolve()):
            raise ValueError("Unsafe archive path")
    z.extractall(destination)
root = next(destination.rglob("pipeline/ingest_v2.py")).parent.parent
os.chdir(root)
print("Project:", root)

In [ ]:
%pip install -q -r requirements.txt
import subprocess, sys
def run(script):
    subprocess.run([sys.executable, script], check=True)

## Update through the latest completed UTC day
This can take time and may pause at an API quota window. Re-run after a failure; cached requests are reused. Save the extracted folder to Google Drive if you need to resume in another Colab session.

In [ ]:
run("pipeline/ingest_v2.py")
run("pipeline/analyze.py")
run("pipeline/refresh_current.py")
run("pipeline/eda.py")
run("pipeline/validate_v2.py")

## Download one daily historical CSV
Contains all districts, weather variables, discharge, accumulated rainfall, source labels and derived percentiles. The live/provider forecast file is separate to avoid treating predictions as observations.

In [ ]:
files.download("dist/data/historical-daily.csv")
files.download("dist/data/forecast-daily.csv")

## Optional: combine genuine hourly weather into one compressed CSV
River discharge is daily and stays in the daily export. It is not repeated 24 times to inflate the hourly data count.

In [ ]:
import pandas as pd, gzip, json
ids = {l["id"] for l in json.load(open("pipeline/locations.json"))}
paths = sorted(p for p in pathlib.Path("data/processed-v2/hourly").glob("year=*/*.csv.gz") if p.name.removesuffix(".csv.gz") in ids)
output = "Sri-Lanka-hourly-weather-2016-latest.csv.gz"
with gzip.open(output, "wt") as out:
    for i, path in enumerate(paths):
        pd.read_csv(path).to_csv(out, index=False, header=(i == 0))
files.download(output)